# Solar filament instance segmentation: from the committed weights to `submission.csv`

This notebook reproduces our submission to the *Solar Filament Segmentation Challenge 2026* (Kaggle competition `filament-segmentation-2026`, metric: Panoptic Quality) from one U-Net in `weights/` and the keep rule in `filseg/rule.json`. It needs no training. `MODEL = "final1"` (default) is the U-Net trained on all 707 training images; `"final2"` is a model trained on four of the five folds. Every computation is a call into the `filseg` package; the cells only call functions, show results and plot.

| step | what happens | code |
|---|---|---|
| 1 | one image before and after normalisation (z-score over the solar disk) | `filseg.data.normalize` |
| 2 | the chosen U-Net (`MODEL`) and the keep rule are loaded (fp16 file, no download) | `filseg.infer.load_models`, `load_rule` |
| 3 | every test image: 4 flip views of the model, averaged, components of the probability map, keep rule | `filseg.infer.segment` |
| 4 | the kept components are written as column-major RLE and validated | `filseg.instances.write_submission`, `filseg.preflight.validate_submission` |
| 5 | masks drawn over an image | `filseg.preflight.read_submission` |
| 6 | fold 0 scored out-of-fold with the host-exact PQ by the fold-0 model (only if the annotation file is present) | `filseg.pq.score` |
| 7 | the training commands (run only if `RUN_TRAINING = True`) | `filseg.job` |

**To run it.** Install the pinned packages with `pip install -r requirements.txt` and open the notebook from the repository folder (the folder that holds `filseg/`). Set `DATA_DIR` in the next cell if the competition data is not found automatically. With a T4 GPU one image takes about a second at 2048 px, so the 180 test images take a few minutes; a CPU is much slower, so for a check set `N_IMAGES = 2` (or the environment variable `FILSEG_N_IMAGES=2`). `README.md` has the Kaggle and local recipes.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import torch

# ----- settings: edit these ----------------------------------------------------------------------
DATA_DIR = os.environ.get("FILSEG_DATA")        # folder that holds train/ and test/; None = autodetect (filseg/data.py)
MODEL = os.environ.get("FILSEG_MODEL", "final1")                              # "final1": trained on all 707 images; "final2": fold-0 model (weights/manifest.json)
N_IMAGES = int(os.environ["FILSEG_N_IMAGES"]) if os.environ.get("FILSEG_N_IMAGES") else None   # None = all test images; N = first N
RUN_TRAINING = False                            # True = run the training commands of the last cell (needs GPUs)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WEIGHTS_DIR = os.environ.get("FILSEG_WEIGHTS", "weights")        # holds manifest.json and the .pt files
RULE_JSON = os.environ.get("FILSEG_RULE", "filseg/rule.json")    # keep rule written by filseg.tune
SCORE_FOLD = 0                                  # training fold scored in step 6 (the fold-0 model never saw it)
OUT_DIR = Path(os.environ.get("FILSEG_OUT", "."))                # submission.csv is written here
# --------------------------------------------------------------------------------------------------

REPO = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "filseg").is_dir()), None)
if REPO is None:
    raise RuntimeError("start this notebook from the repository folder (the one that holds filseg/)")
sys.path.insert(0, str(REPO))
if DATA_DIR:
    os.environ["FILSEG_DATA"] = str(DATA_DIR)   # filseg.data reads it on import: restart the kernel to change it

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pycocotools import mask as mask_util

from filseg import data as D
from filseg import infer, instances, pq, preflight

STEMS = sorted(p.stem for p in D.TEST_IMG.glob("*.jpeg"))[:N_IMAGES]
if not STEMS:
    raise FileNotFoundError(f"no test images in {D.TEST_IMG}: set DATA_DIR to the folder that holds test/test_images")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"model {MODEL}, device {DEVICE}, data {D.DATA_ROOT}, {len(STEMS)} test images")

## 1. One image before and after normalisation

The network never sees raw brightness. `normalize` subtracts the mean and divides by the standard deviation of the pixels inside 0.98 of the solar radius, so every image has the same scale whatever the exposure, and it sets the sky outside the disk to 0.

In [ ]:
stem = STEMS[0]
img = D.load_image(D.TEST_IMG / f"{stem}.jpeg")      # uint8, 2048 x 2048, one channel
x = D.normalize(img)                                 # float32 z-score; 0 outside the disk
disk = x[x != 0]

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].imshow(img, cmap="gray")
ax[0].set_title(f"{stem}: raw uint8")
ax[1].imshow(x, cmap="gray", vmin=-3, vmax=3)
ax[1].set_title("normalised (shown clipped to +-3)")
ax[2].hist(disk, bins=100)
ax[2].set_title(f"all disk pixels incl. limb ring: mean {disk.mean():.2f}, std {disk.std():.2f}")
for a in ax[:2]:
    a.axis("off")
plt.tight_layout()
plt.show()

## 2. Load the model and the keep rule

`weights/manifest.json` lists the committed models (run, fold, encoder, working resolution, SHA-256) and tags two of them `final1` and `final2`. `load_models` takes the one entry named by `MODEL`, checks its file against the hash and rebuilds the U-Net from the fp16 `state_dict`; it never downloads anything and it never averages models. The keep rule (`filseg/rule.json`) was tuned on out-of-fold predictions.

In [ ]:
MEMBERS = infer.load_models(WEIGHTS_DIR, DEVICE, tag=MODEL)     # exactly one (manifest entry, model)
RULE = infer.load_rule(RULE_JSON)
for e, _ in MEMBERS:
    print(f"{MODEL}: {e['file']:<14} run {e['run']}  fold {e['fold']}  {e['encoder']}  work_res {e['work_res']}  sha256 {e['sha256'][:12]}")
print(RULE)

## 3. Predict the test set, then components and the keep rule

For each image, the model predicts four views (as is, left-right, up-down, both); the views are flipped back and averaged into one probability map at 2048 px. Its 8-connected components above `t_lo` are the candidate filaments; the rule keeps a candidate when its peak and mean probability and its area are high enough and it lies inside `max_r` of the solar radius. All of this is `infer.segment`. The loop keeps only the RLE masks, plus everything of the first image for the figures below.

In [ ]:
t0, preds, demo = time.time(), {}, None
for i, stem in enumerate(STEMS, 1):
    seg = infer.segment(MEMBERS, D.load_image(D.TEST_IMG / f"{stem}.jpeg"), RULE, DEVICE)
    preds[stem] = seg.rles
    if i == 1:
        demo = seg
    if i <= 2 or i % 10 == 0 or i == len(STEMS):
        print(f"{i}/{len(STEMS)} {stem}: {len(seg.cands)} components, {len(seg.kept)} kept, {(time.time() - t0) / i:.1f} s/image")
n_masks = sum(map(len, preds.values()))
print(f"{n_masks} masks on {len(preds)} images ({n_masks / len(preds):.2f} per image), {time.time() - t0:.0f} s in total")

In [ ]:
# the first image: every candidate component with the features the rule looks at
cols = ["id", "area", "peak", "mean", "r_frac"]
cands = pd.DataFrame(demo.cands, columns=cols).set_index("id")
cands["kept"] = cands.index.isin(demo.kept)
print(f"{STEMS[0]}: {len(cands)} components above t_lo = {RULE.t_lo}, {int(cands.kept.sum())} kept "
      f"(peak >= {RULE.peak}, mean >= {RULE.mean}, area >= {RULE.min_area}, r/R <= {RULE.max_r})")
cands.sort_values("area", ascending=False).head(12).round(3)

## 4. Write and validate `submission.csv`

One row per kept component: `filament_id` is `<image stem>_<k>` with k counting from 1, `segmentation_rle` is the run-length encoding in column-major order (the host's convention; a row-major file would transpose every mask). The masks of one image share no pixel, because they come from one label map. `validate_submission` raises on any format fault or overlap, and the round trip proves the file decodes to exactly the masks held in memory.

In [ ]:
CSV = OUT_DIR / "submission.csv"
instances.write_submission(preds, CSV)
preflight.validate_submission(CSV, set(STEMS))                    # ValueError on any format fault or overlap
assert preflight.roundtrip_equal(CSV, preds), "the CSV does not decode to the masks in memory"
print(f"{CSV.resolve()}: {n_masks} masks on {len(preds)} images, sha256 {preflight.sha256_file(CSV)[:16]}")
for line in CSV.read_text(encoding="utf-8").splitlines()[:3]:
    print(line[:100] + (" ..." if len(line) > 100 else ""))

## 5. Masks over an image

Left: the probability map of the first test image; green outlines are the components the rule kept, orange ones are components it rejected. Right: the masks read back from `submission.csv` over the image.

In [ ]:
COLOURS = [(255, 60, 60), (60, 255, 60), (60, 160, 255), (255, 220, 0), (255, 60, 255), (0, 230, 230)]


def outline(canvas, mask, colour, width):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(canvas, contours, -1, colour, width)


stem = STEMS[0]
masks = [mask_util.decode(r).astype(bool) for r in preflight.read_submission(CSV).get(stem, [])]
shown = cv2.cvtColor(D.load_image(D.TEST_IMG / f"{stem}.jpeg"), cv2.COLOR_GRAY2RGB)
for k, m in enumerate(masks):
    colour = COLOURS[k % len(COLOURS)]
    shown[m] = (0.5 * shown[m] + 0.5 * np.array(colour)).astype(np.uint8)
    outline(shown, m, colour, 3)
pmap = cv2.cvtColor(np.uint8(np.round(demo.prob * 255)), cv2.COLOR_GRAY2RGB)
for c in demo.cands:
    outline(pmap, demo.labels == c["id"], (60, 255, 60) if c["id"] in demo.kept else (255, 140, 0), 2)

fig, ax = plt.subplots(1, 2, figsize=(16, 8))
ax[0].imshow(pmap)
ax[0].set_title(f"{stem}: probability map, kept (green) and rejected (orange) components")
ax[1].imshow(shown)
ax[1].set_title(f"{stem}: {len(masks)} masks from submission.csv")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 6. Score fold 0

If the annotation file is present, this cell scores the held-out images of fold 0 with `pq.score`, the re-implementation of the host's metric that matches the host's scorer. It uses the fold-0 model (`final2`), which never saw those images, so the result is out-of-fold (the `final1` model saw them, so it is not used here). The keep rule was tuned on the out-of-fold predictions of all five folds, so this number is somewhat optimistic; the nested estimate in the README is the unbiased one. With `N_IMAGES` set, only the first N images of the fold are scored.

In [ ]:
if not D.TRAIN_JSON.exists():
    print(f"skipped: no annotation file at {D.TRAIN_JSON}")
else:
    score_members = MEMBERS if MEMBERS[0][0]["fold"] == SCORE_FOLD else infer.load_models(WEIGHTS_DIR, DEVICE, tag="final2")
    assert score_members[0][0]["fold"] == SCORE_FOLD, "the final2 model must be the fold-0 model"
    folds = json.loads(D.FOLDS_JSON.read_text(encoding="utf-8"))
    val_stems = sorted(s for s, k in folds.items() if k == SCORE_FOLD)[:N_IMAGES]
    records = D.load_records()                    # also checks the SHA-256 of the annotation file
    truth = {rid: rles for s in val_stems for rid, rles in records[s].items()}
    val_preds = {s: infer.segment(score_members, D.load_image(D.TRAIN_IMG / f"{s}.jpeg"), RULE, DEVICE).rles for s in val_stems}
    res = pq.score(truth, val_preds)
    print(f"fold {SCORE_FOLD}, {len(val_stems)} held-out images, model {score_members[0][0]['file']}: "
          f"PQ {res['pq']:.4f}  SQ {res['sq']:.4f}  RQ {res['rq']:.4f}  (TP {res['tp']}, FP {res['fp']}, FN {res['fn']})")

## 7. Training commands (optional)

The committed weights come from these commands; they are not needed to reproduce the CSV. `A:0` to `A:4` train the five fold models of run `A` (used for the out-of-fold maps, the keep rule and `final2`); `AF:5` trains `final1` on all images (fold index 5 means every training image). Each `filseg.job` call trains and predicts one job per GPU (a Kaggle session has two T4s). The cell prints the commands and runs them only if `RUN_TRAINING` is `True` and a GPU is present. Outputs go to `OUT_DIR/runs/<run>/` and never overwrite `weights/`. README, section "Retrain on Kaggle", shows the Kaggle route and how to turn new outputs into committed weights.

In [ ]:
GPUS = torch.cuda.device_count() or 2             # a Kaggle session has two T4s
ALL = [f"A:{k}" for k in range(5)] + ["AF:5"]
JOBS = [ALL[i:i + GPUS] for i in range(0, len(ALL), GPUS)]
for jobs in JOBS:
    print("python -m filseg.job " + " ".join(jobs))
if RUN_TRAINING:
    if torch.cuda.device_count() < 1:
        raise RuntimeError("training needs a GPU: use Kaggle (see README, 'Retrain on Kaggle')")
    env = {**os.environ, "FILSEG_RUNS": str((OUT_DIR / "runs").resolve())}
    for jobs in JOBS:
        subprocess.run([sys.executable, "-m", "filseg.job", *jobs], cwd=REPO, env=env, check=True)